A단계 코드

필요 컬럼 추출 , 조인, 도메인별 데이터 셋 분류 작업

In [2]:
import os
import pandas as pd
import numpy as np

# =========================================================
# 0) 원본 로드 
#    - 조인키/식별키는 문자열로 강제
# =========================================================
dtype_keys = {
    "click_key": "string",
    "ads_idx": "string",
    "mda_idx": "string",
    "pub_sub_rel_id": "string",
}

df_p = pd.read_csv(
    "/Users/joshuakim/Desktop/최종 프로젝트 코드 모음/아이브 코리아 데이터/df_engagement_cleaned.csv",
    dtype=dtype_keys
)

df_save = pd.read_csv(
    '/Users/joshuakim/Desktop/최종 프로젝트 코드 모음/아이브 코리아 데이터/df_reward_cleaned.csv',
    dtype=dtype_keys
)

df_list = pd.read_csv(
    '/Users/joshuakim/Desktop/최종 프로젝트 코드 모음/아이브 코리아 데이터/광고 목록_도메인 라벨링_결측치처리.csv',
    dtype=dtype_keys
)

print("원본 로드 완료")
print("df_p   :", df_p.shape)
print("df_save:", df_save.shape)
print("df_list:", df_list.shape)

# =========================================================
# 1) 필요한 컬럼만 추출 (슬림 컬럼)
#    - KPI/조인/세그먼트에 필요한 최소 + 설명용 일부
# =========================================================

# 광고참여: 중심 테이블
cols_p_required = ["click_key", "ads_idx", "mda_idx", "pub_sub_rel_id"]
cols_p_optional = ["click_date", "click_day", "click_time"]  # 있으면 유지 가능

# 광고적립: 전환/정산
cols_save_required = ["click_key", "show_cost", "earn_cost", "rwd_cost"]
cols_save_optional = ["adv_cost", "ctit", "regdate"]  # 있으면 참고용으로 유지

# 광고목록: 도메인/유형/계약기준
cols_list_required = ["ads_idx", "domain_label", "ads_type"]
cols_list_optional = ["ads_name", "ads_contract_price", "ads_reward_price", "ads_day_cap", "ads_category"]

def pick_existing_cols(df, required_cols, optional_cols=None):
    optional_cols = optional_cols or []
    req = [c for c in required_cols if c in df.columns]
    opt = [c for c in optional_cols if c in df.columns]
    cols = list(dict.fromkeys(req + opt))
    return cols

p_cols = pick_existing_cols(df_p, cols_p_required, cols_p_optional)
save_cols = pick_existing_cols(df_save, cols_save_required, cols_save_optional)
list_cols = pick_existing_cols(df_list, cols_list_required, cols_list_optional)

# 필수 컬럼 누락 체크
missing_p = [c for c in cols_p_required if c not in df_p.columns]
missing_save = [c for c in cols_save_required if c not in df_save.columns]
missing_list = [c for c in cols_list_required if c not in df_list.columns]

if missing_p:
    raise ValueError(f"df_p 필수 컬럼 누락: {missing_p}")
if missing_save:
    raise ValueError(f"df_save 필수 컬럼 누락: {missing_save}")
if missing_list:
    raise ValueError(f"df_list 필수 컬럼 누락: {missing_list}")

df_p_slim = df_p[p_cols].copy()
df_save_slim = df_save[save_cols].copy()
df_list_slim = df_list[list_cols].copy()

print("\n슬림 컬럼 추출 완료")
print("df_p_slim   :", df_p_slim.shape, p_cols)
print("df_save_slim:", df_save_slim.shape, save_cols)
print("df_list_slim:", df_list_slim.shape, list_cols)

# =========================================================
# 2) 슬림 CSV 저장
# =========================================================
OUT_DIR = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\분석에 사용할 전처리 완료 최종 데이터 셋\슬림_테이블 버전 2"
os.makedirs(OUT_DIR, exist_ok=True)

p_slim_path = os.path.join(OUT_DIR, "df_p_slim.csv")
save_slim_path = os.path.join(OUT_DIR, "df_save_slim.csv")
list_slim_path = os.path.join(OUT_DIR, "df_list_slim.csv")

df_p_slim.to_csv(p_slim_path, index=False, encoding="utf-8-sig")
df_save_slim.to_csv(save_slim_path, index=False, encoding="utf-8-sig")
df_list_slim.to_csv(list_slim_path, index=False, encoding="utf-8-sig")

print("\n슬림 CSV 저장 완료")
print(p_slim_path)
print(save_slim_path)
print(list_slim_path)

# =========================================================
# 3) 조인(event 테이블 생성)
#    event = df_p_slim LEFT JOIN df_list_slim(on=ads_idx)
#                    LEFT JOIN df_save_slim(on=click_key)
# =========================================================

# --- 조인키 타입 재강제(혹시 저장/로드 과정 없이 바로 써도 안전)
for c in ["click_key", "ads_idx", "mda_idx", "pub_sub_rel_id"]:
    if c in df_p_slim.columns:
        df_p_slim[c] = df_p_slim[c].astype("string")

for c in ["click_key"]:
    if c in df_save_slim.columns:
        df_save_slim[c] = df_save_slim[c].astype("string")

for c in ["ads_idx"]:
    if c in df_list_slim.columns:
        df_list_slim[c] = df_list_slim[c].astype("string")

# --- (선택) 중복 키 체크: 행증가 위험 감지용
print("\n[중복 체크]")
if "click_key" in df_save_slim.columns:
    dup_save = df_save_slim["click_key"].duplicated(keep=False).sum()
    print(f"df_save_slim click_key 중복 행 수: {dup_save}")

if "ads_idx" in df_list_slim.columns:
    dup_list = df_list_slim["ads_idx"].duplicated(keep=False).sum()
    print(f"df_list_slim ads_idx 중복 행 수: {dup_list}")

# --- df_save 중복 click_key 처리 (필요 시)
# 1,2번 문제 없다고 하셨지만, 안전하게 옵션 코드 남김
# 중복이 있다면 click_key 기준 집계해서 1행으로 만든 뒤 조인
if df_save_slim["click_key"].duplicated().any():
    num_cols = [c for c in ["show_cost", "earn_cost", "rwd_cost", "adv_cost", "ctit"] if c in df_save_slim.columns]
    non_num_cols = [c for c in df_save_slim.columns if c not in (["click_key"] + num_cols)]

    # 숫자형 변환(실패 시 NaN)
    for c in num_cols:
        df_save_slim[c] = pd.to_numeric(df_save_slim[c], errors="coerce")

    agg_dict = {c: "sum" for c in num_cols if c != "ctit"}
    if "ctit" in num_cols:
        agg_dict["ctit"] = "median"  # ctit는 대표값으로 중앙값 사용
    for c in non_num_cols:
        agg_dict[c] = "first"

    df_save_join = (
        df_save_slim
        .groupby("click_key", as_index=False)
        .agg(agg_dict)
    )
    print(f"df_save_slim 중복 click_key 존재 → 집계 후 조인용 df_save_join 사용: {df_save_join.shape}")
else:
    df_save_join = df_save_slim.copy()
    # 숫자형 컬럼 캐스팅
    for c in ["show_cost", "earn_cost", "rwd_cost", "adv_cost", "ctit"]:
        if c in df_save_join.columns:
            df_save_join[c] = pd.to_numeric(df_save_join[c], errors="coerce")

# --- df_list 중복 ads_idx 처리 (필요 시)
if df_list_slim["ads_idx"].duplicated().any():
    # 우선 첫 행 유지 (필요하면 최신/비결측 우선 로직으로 교체)
    df_list_join = df_list_slim.drop_duplicates(subset=["ads_idx"], keep="first").copy()
    print(f"df_list_slim 중복 ads_idx 존재 → 첫 행 기준 dedup 후 조인용 df_list_join 사용: {df_list_join.shape}")
else:
    df_list_join = df_list_slim.copy()

# --- 조인
event = (
    df_p_slim
    .merge(df_list_join, on="ads_idx", how="left", validate="m:1")
    .merge(df_save_join, on="click_key", how="left", validate="m:1")
)

print("\n[조인 완료] event shape:", event.shape)

# =========================================================
# 3-1) 전환 플래그/수익 파생변수 생성 (정의 반영)
# =========================================================

# df_save에 행이 있으면 전환(적립 발생)
# 비용컬럼 결측 여부가 아닌, click_key 매칭 여부를 우선으로 보기 위해
# 조인 후 show_cost/earn_cost/rwd_cost 중 하나라도 존재하면 매칭된 것으로 간주 가능
# (추가로 _merge를 쓰는 방법도 있으나 여기선 간단히 처리)
save_cols_presence = [c for c in ["show_cost", "earn_cost", "rwd_cost", "adv_cost", "ctit"] if c in event.columns]

if save_cols_presence:
    event["is_conv"] = event[save_cols_presence].notna().any(axis=1).astype(int)
else:
    event["is_conv"] = 0

# 숫자형 보정
for c in ["show_cost", "earn_cost", "rwd_cost"]:
    if c in event.columns:
        event[c] = pd.to_numeric(event[c], errors="coerce")

# 정의: 아이브 수익(Profit) = show_cost - earn_cost
event["profit"] = event["show_cost"].fillna(0) - event["earn_cost"].fillna(0)

# =========================================================
# 3-2) 음수 profit 존재 여부 확인 (요청)
# =========================================================
neg_profit_rows = event["profit"] < 0
neg_cnt = int(neg_profit_rows.sum())
neg_ratio = (neg_cnt / len(event) * 100) if len(event) else 0

print("\n[profit 음수 체크]")
print(f"음수 profit 행 수: {neg_cnt:,} ({neg_ratio:.4f}%)")

if neg_cnt > 0:
    print("※ 음수 profit 샘플 5건")
    sample_cols = [c for c in ["click_key", "ads_idx", "mda_idx", "pub_sub_rel_id", "show_cost", "earn_cost", "profit"] if c in event.columns]
    print(event.loc[neg_profit_rows, sample_cols].head(5))

# =========================================================
# 3-3) event CSV 저장 (도메인 분리 전 통합본)
# =========================================================
event_out_path = os.path.join(OUT_DIR, "event_joined_before_domain_split.csv")
event.to_csv(event_out_path, index=False, encoding="utf-8-sig")
print("\n통합 event 저장 완료:", event_out_path)

# =========================================================
# 3-4) 간단 검증 출력
# =========================================================
print("\n[간단 검증]")
print("총 클릭수(행수):", len(event))
print("총 전환수(is_conv=1):", int(event["is_conv"].sum()))
print("총 Profit:", float(event["profit"].fillna(0).sum()))
print("CVR(전체):", float(event["is_conv"].mean()) if len(event) else np.nan)

원본 로드 완료
df_p   : (1048574, 16)
df_save: (939745, 15)
df_list: (445260, 26)

슬림 컬럼 추출 완료
df_p_slim   : (1048574, 7) ['click_key', 'ads_idx', 'mda_idx', 'pub_sub_rel_id', 'click_date', 'click_day', 'click_time']
df_save_slim: (939745, 7) ['click_key', 'show_cost', 'earn_cost', 'rwd_cost', 'adv_cost', 'ctit', 'regdate']
df_list_slim: (445260, 8) ['ads_idx', 'domain_label', 'ads_type', 'ads_name', 'ads_contract_price', 'ads_reward_price', 'ads_day_cap', 'ads_category']

슬림 CSV 저장 완료
D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\분석에 사용할 전처리 완료 최종 데이터 셋\슬림_테이블 버전 2/df_p_slim.csv
D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\분석에 사용할 전처리 완료 최종 데이터 셋\슬림_테이블 버전 2/df_save_slim.csv
D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\분석에 사용할 전처리 완료 최종 데이터 셋\슬림_테이블 버전 2/df_list_slim.csv

[중복 체크]
df_save_slim click_key 중복 행 수: 0
df_list_slim ads_idx 중복 행 수: 0

[조인 완료] event shape: (1048574, 20)

[profit 음수 체크]
음수 profit 행 수: 0 (0.0000%)

통합 event 저장 완료: D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\분석에 사용할 전처리 완료 최종 데이터 셋\슬림_테이블 버전 2/event_

도메인별 분류 테이블 만드는 중

In [3]:
df_join = pd.read_csv('/Users/joshuakim/Desktop/최종 프로젝트 코드 모음/준성님 코드/event_joined_before_domain_split.csv')

In [4]:
df_domain_1 = df_join[df_join['domain_label'] == 1]
df_domain_2 = df_join[df_join['domain_label'] == 2]
df_domain_3 = df_join[df_join['domain_label'] == 3]
df_domain_4 = df_join[df_join['domain_label'] == 4]
df_domain_5 = df_join[df_join['domain_label'] == 5]

In [5]:
df_domain_1.to_csv(r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\분석에 사용할 전처리 완료 최종 데이터 셋\도메인별 분류 테이블\domain_1.csv", index=False, encoding="utf-8-sig")
df_domain_2.to_csv(r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\분석에 사용할 전처리 완료 최종 데이터 셋\도메인별 분류 테이블\domain_2.csv", index=False, encoding="utf-8-sig")
df_domain_3.to_csv(r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\분석에 사용할 전처리 완료 최종 데이터 셋\도메인별 분류 테이블\domain_3.csv", index=False, encoding="utf-8-sig")
df_domain_4.to_csv(r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\분석에 사용할 전처리 완료 최종 데이터 셋\도메인별 분류 테이블\domain_4.csv", index=False, encoding="utf-8-sig")
df_domain_5.to_csv(r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\분석에 사용할 전처리 완료 최종 데이터 셋\도메인별 분류 테이블\domain_5.csv", index=False, encoding="utf-8-sig")

B단계 코드

본격 매체사 관리 시스템 코드

최종 버전 3 볼륨형 상위 조건 완화/ 프리미엄형 확정 추천에 추가 규모 조건

+ B단계 최종 domain_label × ads_type × mda_idx × ads_idx 단위 테이블 생성
+ DOMAIN_NO에 도메인 라벨별로 넣어서 돌려줘야 함. 즉 5번 돌려야함

In [6]:
import os
import numpy as np
import pandas as pd

# =========================================================
# 0) 설정
# =========================================================
BASE_DIR = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\분석에 사용할 전처리 완료 최종 데이터 셋\도메인별 분류 테이블"
DOMAIN_NO = 4
DOMAIN_FILE = os.path.join(BASE_DIR, f"domain_{DOMAIN_NO}.csv")

OUT_ROOT = os.path.join(BASE_DIR, f"outputs_도메인 {DOMAIN_NO}_ads_idx단위 상위메인_최종 v4")
DOMAIN_OUT = os.path.join(OUT_ROOT, f"domain{DOMAIN_NO}")
os.makedirs(DOMAIN_OUT, exist_ok=True)

ads_type_map = {
    1: "설치형", 2: "실행형", 3: "참여형", 4: "클릭형",
    5: "페북", 6: "트위터", 7: "인스타", 8: "노출형",
    9: "퀘스트", 10: "유튜브", 11: "네이버", 12: "CPS(물건구매)"
}

# 광고유형별 최소 CVR 기준
cvr_threshold_by_type = {
    1: 0.50,
    2: 0.50,
    3: 0.20,
    4: 0.20,
    5: 0.20,
    6: 0.20,
    7: 0.20,
    8: 0.20,
    9: 0.20,
    10: 0.20,
    11: 0.20,
    12: 0.10
}

# =========================================================
# 1) 퍼센타일 기준
# =========================================================
PREMIUM_PPC_Q = 0.70
PREMIUM_MR_Q  = 0.60

VOLUME_PROFIT_Q = 0.70
VOLUME_CONV_Q   = 0.70
VOLUME_PPC_Q    = 0.50
VOLUME_MR_Q     = 0.50

IMPROVE_FAIL_Q = 0.50

RECOMMEND_SCALE_Q = 0.50

MDA_PROFIT_RETENTION_MIN = 0.95
MDA_CONV_RETENTION_MIN   = 0.90
MDA_SMALL_N_TYPE         = 10

LOW_CLICK_PUBSUB_Q = 0.25


# =========================================================
# 2) 공통 유틸
# =========================================================
def load_domain_event(path: str) -> pd.DataFrame:
    dtype_keys = {
        "click_key": "string",
        "ads_idx": "string",
        "ads_name": "string",
        "mda_idx": "string",
        "pub_sub_rel_id": "string",
        "domain_label": "string",
        "ads_type": "string",
    }
    df = pd.read_csv(path, dtype=dtype_keys)

    for c in ["show_cost", "earn_cost", "rwd_cost", "profit", "is_conv"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    if "profit" not in df.columns:
        df["profit"] = df["show_cost"].fillna(0) - df["earn_cost"].fillna(0)

    if "is_conv" not in df.columns:
        save_presence = [c for c in ["show_cost", "earn_cost", "rwd_cost"] if c in df.columns]
        if save_presence:
            df["is_conv"] = df[save_presence].notna().any(axis=1).astype(int)
        else:
            df["is_conv"] = 0

    df["ads_type"] = pd.to_numeric(df["ads_type"], errors="coerce").astype("Int64")
    df["domain_label"] = pd.to_numeric(df["domain_label"], errors="coerce").astype("Int64")

    need = [
        "click_key", "ads_idx", "ads_name", "mda_idx", "pub_sub_rel_id",
        "ads_type", "show_cost", "earn_cost", "is_conv", "profit"
    ]
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise ValueError(f"필수 컬럼 누락: {miss}")

    return df


def add_cvr_threshold_and_pass(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["cvr_threshold"] = out["ads_type"].map(cvr_threshold_by_type)
    out["cvr_pass"] = (out["cvr"] >= out["cvr_threshold"]).astype("Int64")
    return out


def make_candidates_from_dist(s: pd.Series, base_list: list) -> list:
    s = pd.to_numeric(s, errors="coerce").dropna()
    cand = set(base_list)

    if len(s) > 0:
        cand.add(int(np.floor(s.quantile(0.25))))
        cand.add(int(np.floor(s.quantile(0.50))))

    cand = [c for c in cand if c is not None and c >= 0]
    return sorted(set(cand))


def safe_quantile(series: pd.Series, q: float):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan
    return s.quantile(q)


# =========================================================
# 3) 상위매체 KPI 직접 집계
#    grain: domain_label × ads_type × mda_idx
# =========================================================
def build_mda_kpi(df: pd.DataFrame) -> pd.DataFrame:
    grp = ["domain_label", "ads_type", "mda_idx"]

    mda = (
        df.groupby(grp, dropna=False)
          .agg(
              clicks=("click_key", "count"),
              conversions=("is_conv", "sum"),
              revenue=("show_cost", "sum"),
              partner_payout=("earn_cost", "sum"),
              reward_cost=("rwd_cost", "sum"),
              profit=("profit", "sum"),
              n_pub_sub=("pub_sub_rel_id", "nunique")
          )
          .reset_index()
    )

    mda["cvr"] = mda["conversions"] / mda["clicks"].replace(0, np.nan)
    mda["revenue_per_click"] = mda["revenue"] / mda["clicks"].replace(0, np.nan)
    mda["profit_per_click"] = mda["profit"] / mda["clicks"].replace(0, np.nan)
    mda["profit_per_conv"] = mda["profit"] / mda["conversions"].replace(0, np.nan)
    mda["margin_rate"] = mda["profit"] / mda["revenue"].replace(0, np.nan)
    mda["ads_type_name"] = mda["ads_type"].map(ads_type_map)

    mda = add_cvr_threshold_and_pass(mda)
    return mda


def mda_min_remaining_n(n0: int) -> int:
    return max(5, int(np.ceil(n0 * 0.10)))


def select_auto_mda_cut_for_type(g: pd.DataFrame, ads_type: int) -> dict:
    n0 = len(g)
    profit0 = g["profit"].fillna(0).sum()
    conv0 = g["conversions"].fillna(0).sum()

    if n0 < MDA_SMALL_N_TYPE:
        return {
            "ads_type": int(ads_type),
            "ads_type_name": ads_type_map.get(int(ads_type)),
            "n0_mda": int(n0),
            "selected_min_mda_conversions": 0,
            "selected_min_mda_profit": 0,
            "profit_retention": 1.0,
            "conv_retention": 1.0,
            "n_retained": int(n0),
            "rule": "SMALL_N_EXCEPTION",
            "note": "mda 수가 적어 신뢰도 컷 완화"
        }

    conv_cand = make_candidates_from_dist(g["conversions"], [0, 1, 3, 5])

    profit_series = pd.to_numeric(g["profit"], errors="coerce").dropna()
    profit_cand = [0]
    if len(profit_series) > 0:
        profit_cand += [
            int(np.floor(profit_series.quantile(0.10))),
            int(np.floor(profit_series.quantile(0.25))),
            int(np.floor(profit_series.quantile(0.50)))
        ]
    profit_cand = sorted(set([x for x in profit_cand if x >= 0]))

    min_n = mda_min_remaining_n(n0)
    feasible = []

    for mc in conv_cand:
        for mp in profit_cand:
            gg = g[(g["conversions"] >= mc) & (g["profit"] >= mp)]
            n1 = len(gg)
            if n1 == 0:
                continue

            pr = gg["profit"].fillna(0).sum() / profit0 if profit0 != 0 else 1.0
            cr = gg["conversions"].fillna(0).sum() / conv0 if conv0 != 0 else 1.0

            ok = (
                (pr >= MDA_PROFIT_RETENTION_MIN) and
                (cr >= MDA_CONV_RETENTION_MIN) and
                (n1 >= min_n)
            )
            if ok:
                feasible.append((mc, mp, pr, cr, n1))

    if feasible:
        feasible_sorted = sorted(feasible, key=lambda x: (x[0], x[1], x[4]), reverse=True)
        mc, mp, pr, cr, n1 = feasible_sorted[0]
        return {
            "ads_type": int(ads_type),
            "ads_type_name": ads_type_map.get(int(ads_type)),
            "n0_mda": int(n0),
            "selected_min_mda_conversions": int(mc),
            "selected_min_mda_profit": int(mp),
            "profit_retention": float(pr),
            "conv_retention": float(cr),
            "n_retained": int(n1),
            "rule": "FEASIBLE_MOST_STRINGENT",
            "note": f"가드레일 충족(Profit≥{MDA_PROFIT_RETENTION_MIN}, Conv≥{MDA_CONV_RETENTION_MIN}, n≥{min_n})"
        }

    return {
        "ads_type": int(ads_type),
        "ads_type_name": ads_type_map.get(int(ads_type)),
        "n0_mda": int(n0),
        "selected_min_mda_conversions": 0,
        "selected_min_mda_profit": 0,
        "profit_retention": 1.0,
        "conv_retention": 1.0,
        "n_retained": int(n0),
        "rule": "NO_FEASIBLE_EXCEPTION",
        "note": "조건 만족 mda 컷 없음 → 예외 처리"
    }


def build_auto_mda_cut_table(mda_kpi: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for ads_type, g in mda_kpi.groupby("ads_type", dropna=False):
        if pd.isna(ads_type):
            continue
        rows.append(select_auto_mda_cut_for_type(g, int(ads_type)))
    return pd.DataFrame(rows)


def apply_mda_reliable(mda_kpi: pd.DataFrame, mda_cut_table: pd.DataFrame) -> pd.DataFrame:
    df = mda_kpi.merge(
        mda_cut_table[["ads_type", "selected_min_mda_conversions", "selected_min_mda_profit"]],
        on="ads_type", how="left"
    )
    df["selected_min_mda_conversions"] = df["selected_min_mda_conversions"].fillna(0).astype(int)
    df["selected_min_mda_profit"] = df["selected_min_mda_profit"].fillna(0).astype(int)

    df["mda_reliable"] = (
        (df["conversions"] >= df["selected_min_mda_conversions"]) &
        (df["profit"] >= df["selected_min_mda_profit"])
    ).astype(int)

    return df


# =========================================================
# 4) 광고 ID 기준 정합성 검증
# =========================================================
def validate_ads_master_mapping(df: pd.DataFrame) -> pd.DataFrame:
    check = (
        df.groupby("ads_idx", dropna=False)
          .agg(
              n_ads_name=("ads_name", lambda x: x.nunique(dropna=False)),
              n_ads_type=("ads_type", lambda x: x.nunique(dropna=False)),
              n_mda_idx=("mda_idx", lambda x: x.nunique(dropna=False))
          )
          .reset_index()
    )

    issues = check[
        (check["n_ads_name"] > 1) |
        (check["n_ads_type"] > 1) |
        (check["n_mda_idx"] > 1)
    ].copy()

    return issues


# =========================================================
# 5) 성과 세그먼트 + 운영 태그 부여
# =========================================================
def segment_mda_direct(mda_kpi: pd.DataFrame) -> pd.DataFrame:
    out_list = []

    for ads_type, g in mda_kpi.groupby("ads_type", dropna=False):
        g = g.copy()

        ref = g[g["mda_reliable"] == 1].copy()
        if ref.empty:
            ref = g.copy()

        ppc_q70 = safe_quantile(ref["profit_per_conv"], PREMIUM_PPC_Q)
        mr_q60 = safe_quantile(ref["margin_rate"], PREMIUM_MR_Q)

        profit_q70 = safe_quantile(ref["profit"], VOLUME_PROFIT_Q)
        conv_q70 = safe_quantile(ref["conversions"], VOLUME_CONV_Q)

        ppc_q50 = safe_quantile(ref["profit_per_conv"], VOLUME_PPC_Q)
        mr_q50 = safe_quantile(ref["margin_rate"], VOLUME_MR_Q)

        profit_q50_scale = safe_quantile(ref["profit"], RECOMMEND_SCALE_Q)
        conv_q50_scale   = safe_quantile(ref["conversions"], RECOMMEND_SCALE_Q)

        g["cut_ppc_q70"] = ppc_q70
        g["cut_mr_q60"] = mr_q60
        g["cut_profit_q70"] = profit_q70
        g["cut_conv_q70"] = conv_q70
        g["cut_ppc_q50"] = ppc_q50
        g["cut_mr_q50"] = mr_q50
        g["cut_profit_q50_scale"] = profit_q50_scale
        g["cut_conv_q50_scale"] = conv_q50_scale

        perf_segments = []
        perf_reasons = []
        op_tags = []
        op_details = []
        op_reasons = []

        for _, r in g.iterrows():
            if (
                (r["cvr_pass"] == 1) and
                (pd.notna(r["cut_ppc_q70"])) and
                (r["profit_per_conv"] >= r["cut_ppc_q70"]) and
                (pd.notna(r["cut_mr_q60"])) and
                (r["margin_rate"] >= r["cut_mr_q60"])
            ):
                performance_segment = "프리미엄 상위"
                performance_reason = "CVR 기준 충족 + 전환당 이익 상위 + 마진율 우수"

            elif (
                (r["mda_reliable"] == 1) and
                (r["cvr_pass"] == 1) and
                (
                    ((pd.notna(r["cut_profit_q70"])) and (r["profit"] >= r["cut_profit_q70"])) or
                    ((pd.notna(r["cut_conv_q70"])) and (r["conversions"] >= r["cut_conv_q70"]))
                ) and
                (
                    ((pd.notna(r["cut_ppc_q50"])) and (r["profit_per_conv"] >= r["cut_ppc_q50"])) or
                    ((pd.notna(r["cut_mr_q50"])) and (r["margin_rate"] >= r["cut_mr_q50"]))
                )
            ):
                performance_segment = "볼륨형 상위"
                performance_reason = "CVR 기준 충족 + 규모 상위 + 수익성 지표 중 하나 이상 중간 이상"

            elif (
                (
                    ((pd.notna(r["cut_profit_q70"])) and (r["profit"] >= r["cut_profit_q70"])) or
                    ((pd.notna(r["cut_conv_q70"])) and (r["conversions"] >= r["cut_conv_q70"]))
                ) and
                (
                    (r["cvr_pass"] == 0) or
                    ((pd.notna(r["cut_ppc_q50"])) and (r["profit_per_conv"] < r["cut_ppc_q50"])) or
                    ((pd.notna(r["cut_mr_q50"])) and (r["margin_rate"] < r["cut_mr_q50"]))
                )
            ):
                performance_segment = "구조개선형_대형"
                performance_reason = "규모는 크지만 품질 또는 수익성 구조에 개선 필요"

            elif (r["cvr_pass"] == 1):
                performance_segment = "구조개선형"
                performance_reason = "품질은 충족하지만 프리미엄/볼륨형 기준에는 미달"

            else:
                performance_segment = "저효율 상위"
                performance_reason = "CVR 기준 미달 + 규모/수익성도 낮음"

            perf_segments.append(performance_segment)
            perf_reasons.append(performance_reason)

            if performance_segment == "프리미엄 상위":
                premium_scale_ok = (
                    ((pd.notna(r["cut_profit_q50_scale"])) and (r["profit"] >= r["cut_profit_q50_scale"])) or
                    ((pd.notna(r["cut_conv_q50_scale"])) and (r["conversions"] >= r["cut_conv_q50_scale"]))
                )

                if (r["mda_reliable"] == 1) and premium_scale_ok:
                    operation_tag = "확정추천"
                    operation_detail = "프리미엄형"
                    operation_reason = "질과 수익성이 우수하고, 신뢰도 및 추가 규모 조건도 충족"
                else:
                    operation_tag = "잠재추천"
                    operation_detail = "프리미엄형"
                    operation_reason = "질은 우수하지만 아직 확정추천으로 보기엔 신뢰도 또는 규모가 부족"

            elif performance_segment == "볼륨형 상위":
                if r["mda_reliable"] == 1:
                    operation_tag = "확정추천"
                    operation_detail = "볼륨형"
                    operation_reason = "규모가 크고 기본 품질/수익성도 확보"
                else:
                    operation_tag = "잠재추천"
                    operation_detail = "볼륨형"
                    operation_reason = "볼륨 가능성은 있으나 아직 신뢰도 기준 미달"

            elif performance_segment in ["구조개선형_대형", "구조개선형"]:
                operation_tag = "개선우선"
                operation_detail = performance_segment
                operation_reason = "성과는 일부 있으나 구조적 개선이 필요"

            else:
                operation_tag = "축소후보"
                operation_detail = "저효율형"
                operation_reason = "효율/규모/수익성이 모두 약해 축소 또는 정리 검토"

            op_tags.append(operation_tag)
            op_details.append(operation_detail)
            op_reasons.append(operation_reason)

        g["performance_segment"] = perf_segments
        g["performance_reason"] = perf_reasons
        g["operation_tag"] = op_tags
        g["operation_detail"] = op_details
        g["operation_reason"] = op_reasons
        g["gap_to_cvr_threshold"] = g["cvr_threshold"] - g["cvr"]

        out_list.append(g)

    out = pd.concat(out_list, ignore_index=True) if out_list else mda_kpi
    return out


# =========================================================
# 6) 보조 진단 탭
#    grain: domain_label × ads_type × mda_idx × pub_sub_rel_id
# =========================================================
def build_mda_pubsub_detail(df: pd.DataFrame) -> pd.DataFrame:
    grp = ["domain_label", "ads_type", "mda_idx", "pub_sub_rel_id"]

    detail = (
        df.groupby(grp, dropna=False)
          .agg(
              clicks=("click_key", "count"),
              conversions=("is_conv", "sum"),
              revenue=("show_cost", "sum"),
              partner_payout=("earn_cost", "sum"),
              reward_cost=("rwd_cost", "sum"),
              profit=("profit", "sum")
          )
          .reset_index()
    )

    detail["cvr"] = detail["conversions"] / detail["clicks"].replace(0, np.nan)
    detail["profit_per_conv"] = detail["profit"] / detail["conversions"].replace(0, np.nan)
    detail["margin_rate"] = detail["profit"] / detail["revenue"].replace(0, np.nan)
    detail["ads_type_name"] = detail["ads_type"].map(ads_type_map)

    return detail


def build_mda_diagnostic_summary(mda_pubsub_detail: pd.DataFrame) -> pd.DataFrame:
    detail = mda_pubsub_detail.copy()

    low_click_cut_table = (
        detail.groupby(["domain_label", "ads_type"], dropna=False)["clicks"]
              .quantile(LOW_CLICK_PUBSUB_Q)
              .reset_index(name="low_click_cut_p25")
    )
    detail = detail.merge(low_click_cut_table, on=["domain_label", "ads_type"], how="left")

    detail["is_zero_conv_pubsub"] = (detail["conversions"] == 0).astype(int)
    detail["is_low_profit_pubsub"] = (detail["profit"] <= 0).astype(int)
    detail["is_low_click_pubsub"] = (detail["clicks"] < detail["low_click_cut_p25"]).astype(int)

    detail = detail.sort_values(
        ["domain_label", "ads_type", "mda_idx", "profit"],
        ascending=[True, True, True, False]
    )
    detail["pub_rank_within_mda"] = detail.groupby(
        ["domain_label", "ads_type", "mda_idx"], dropna=False
    )["profit"].rank(method="first", ascending=False)

    grp = ["domain_label", "ads_type", "mda_idx"]

    total_profit = (
        detail.groupby(grp, dropna=False)["profit"]
              .sum()
              .reset_index(name="mda_total_profit")
    )

    top1_profit = (
        detail[detail["pub_rank_within_mda"] == 1]
        .groupby(grp, dropna=False)["profit"]
        .sum()
        .reset_index(name="top1_pubsub_profit")
    )

    top3_profit = (
        detail[detail["pub_rank_within_mda"] <= 3]
        .groupby(grp, dropna=False)["profit"]
        .sum()
        .reset_index(name="top3_pubsub_profit")
    )

    summary = (
        detail.groupby(grp, dropna=False)
              .agg(
                  ads_type_name=("ads_type_name", "first"),
                  n_pub_sub=("pub_sub_rel_id", "nunique"),

                  pubsub_clicks_mean=("clicks", "mean"),
                  pubsub_clicks_median=("clicks", "median"),
                  pubsub_clicks_p75=("clicks", lambda x: x.quantile(0.75)),
                  pubsub_clicks_p90=("clicks", lambda x: x.quantile(0.90)),

                  pubsub_conv_mean=("conversions", "mean"),
                  pubsub_conv_median=("conversions", "median"),
                  pubsub_conv_p75=("conversions", lambda x: x.quantile(0.75)),
                  pubsub_conv_p90=("conversions", lambda x: x.quantile(0.90)),

                  pubsub_profit_mean=("profit", "mean"),
                  pubsub_profit_median=("profit", "median"),
                  pubsub_profit_p75=("profit", lambda x: x.quantile(0.75)),
                  pubsub_profit_p90=("profit", lambda x: x.quantile(0.90)),

                  pubsub_cvr_mean=("cvr", "mean"),
                  pubsub_cvr_median=("cvr", "median"),
                  pubsub_cvr_p75=("cvr", lambda x: x.quantile(0.75)),
                  pubsub_cvr_p90=("cvr", lambda x: x.quantile(0.90)),

                  zero_conv_pubsub_share=("is_zero_conv_pubsub", "mean"),
                  low_profit_pubsub_share=("is_low_profit_pubsub", "mean"),
                  low_click_pubsub_share=("is_low_click_pubsub", "mean"),
                  low_click_cut_p25=("low_click_cut_p25", "first")
              )
              .reset_index()
    )

    summary = summary.merge(total_profit, on=grp, how="left")
    summary = summary.merge(top1_profit, on=grp, how="left")
    summary = summary.merge(top3_profit, on=grp, how="left")

    summary["top_pubsub_profit_share"] = summary["top1_pubsub_profit"] / summary["mda_total_profit"].replace(0, np.nan)
    summary["top3_pubsub_profit_share"] = summary["top3_pubsub_profit"] / summary["mda_total_profit"].replace(0, np.nan)

    return summary


# =========================================================
# 7) 광고 단위 상세 파일
#    grain: domain_label × ads_type × mda_idx × ads_idx
# =========================================================
def build_mda_ads_detail(df: pd.DataFrame) -> pd.DataFrame:
    grp = ["domain_label", "ads_type", "mda_idx", "ads_idx"]

    detail = (
        df.groupby(grp, dropna=False)
          .agg(
              ads_name=("ads_name", "first"),
              clicks=("click_key", "count"),
              conversions=("is_conv", "sum"),
              revenue=("show_cost", "sum"),
              partner_payout=("earn_cost", "sum"),
              reward_cost=("rwd_cost", "sum"),
              profit=("profit", "sum"),
              n_pub_sub=("pub_sub_rel_id", "nunique")
          )
          .reset_index()
    )

    detail["cvr"] = detail["conversions"] / detail["clicks"].replace(0, np.nan)
    detail["revenue_per_click"] = detail["revenue"] / detail["clicks"].replace(0, np.nan)
    detail["profit_per_click"] = detail["profit"] / detail["clicks"].replace(0, np.nan)
    detail["profit_per_conv"] = detail["profit"] / detail["conversions"].replace(0, np.nan)
    detail["margin_rate"] = detail["profit"] / detail["revenue"].replace(0, np.nan)
    detail["ads_type_name"] = detail["ads_type"].map(ads_type_map)

    detail = add_cvr_threshold_and_pass(detail)

    return detail


def build_mda_ads_detail_with_segment(
    mda_ads_detail: pd.DataFrame,
    mda_segmented_full: pd.DataFrame
) -> pd.DataFrame:
    cols_to_merge = [
        "domain_label", "ads_type", "mda_idx", "ads_type_name",
        "performance_segment", "performance_reason",
        "operation_tag", "operation_detail", "operation_reason",
        "mda_reliable",
        "selected_min_mda_conversions", "selected_min_mda_profit",
        "cut_ppc_q70", "cut_mr_q60", "cut_profit_q70", "cut_conv_q70",
        "cut_ppc_q50", "cut_mr_q50",
        "cut_profit_q50_scale", "cut_conv_q50_scale",
        "gap_to_cvr_threshold"
    ]
    cols_to_merge = [c for c in cols_to_merge if c in mda_segmented_full.columns]

    out = mda_ads_detail.merge(
        mda_segmented_full[cols_to_merge].drop_duplicates(
            subset=["domain_label", "ads_type", "mda_idx"]
        ),
        on=["domain_label", "ads_type", "mda_idx", "ads_type_name"],
        how="left"
    )
    return out


# =========================================================
# 8) Top5 추출
# =========================================================
def extract_top5_by_performance_segment(mda_seg: pd.DataFrame) -> pd.DataFrame:
    sort_rules = {
        "프리미엄 상위": ["profit_per_conv", "margin_rate", "profit", "conversions"],
        "볼륨형 상위": ["profit", "conversions", "cvr", "profit_per_click"],
        "구조개선형_대형": ["profit", "conversions", "gap_to_cvr_threshold", "margin_rate"],
        "구조개선형": ["cvr", "profit", "profit_per_conv", "margin_rate"],
        "저효율 상위": ["profit", "conversions", "cvr"]
    }

    results = []

    for ads_type_name, g_type in mda_seg.groupby("ads_type_name", dropna=False):
        for perf_seg, g_seg in g_type.groupby("performance_segment", dropna=False):
            if perf_seg not in sort_rules:
                continue

            sort_cols = [c for c in sort_rules[perf_seg] if c in g_seg.columns]
            g_sorted = g_seg.sort_values(sort_cols, ascending=False).head(5).copy()
            g_sorted["rank_within_ads_type_perf_segment"] = range(1, len(g_sorted) + 1)
            results.append(g_sorted)

    out = pd.concat(results, ignore_index=True) if results else pd.DataFrame()

    cols = [
        "ads_type", "ads_type_name",
        "performance_segment", "rank_within_ads_type_perf_segment",
        "mda_idx",
        "operation_tag", "operation_detail",
        "clicks", "conversions", "revenue", "profit",
        "cvr", "profit_per_click", "profit_per_conv", "margin_rate",
        "mda_reliable",
        "performance_reason", "operation_reason"
    ]
    cols = [c for c in cols if c in out.columns]
    return out[cols].copy()


def extract_top5_by_operation_tag(mda_seg: pd.DataFrame) -> pd.DataFrame:
    def get_sort_cols(tag, detail):
        if tag == "확정추천" and detail == "프리미엄형":
            return ["profit_per_conv", "margin_rate", "profit", "conversions"]
        if tag == "잠재추천" and detail == "프리미엄형":
            return ["profit_per_conv", "margin_rate", "profit", "conversions"]
        if tag == "확정추천" and detail == "볼륨형":
            return ["profit", "conversions", "cvr", "profit_per_click"]
        if tag == "잠재추천" and detail == "볼륨형":
            return ["profit", "conversions", "cvr", "profit_per_click"]
        if tag == "개선우선":
            return ["profit", "conversions", "gap_to_cvr_threshold", "margin_rate"]
        if tag == "축소후보":
            return ["profit", "conversions", "cvr"]
        return ["profit", "conversions"]

    results = []

    for ads_type_name, g_type in mda_seg.groupby("ads_type_name", dropna=False):
        for (op_tag, op_detail), g_seg in g_type.groupby(["operation_tag", "operation_detail"], dropna=False):
            sort_cols = [c for c in get_sort_cols(op_tag, op_detail) if c in g_seg.columns]
            g_sorted = g_seg.sort_values(sort_cols, ascending=False).head(5).copy()
            g_sorted["rank_within_ads_type_operation"] = range(1, len(g_sorted) + 1)
            results.append(g_sorted)

    out = pd.concat(results, ignore_index=True) if results else pd.DataFrame()

    cols = [
        "ads_type", "ads_type_name",
        "operation_tag", "operation_detail", "rank_within_ads_type_operation",
        "performance_segment",
        "mda_idx",
        "clicks", "conversions", "revenue", "profit",
        "cvr", "profit_per_click", "profit_per_conv", "margin_rate",
        "mda_reliable",
        "performance_reason", "operation_reason"
    ]
    cols = [c for c in cols if c in out.columns]
    return out[cols].copy()


# =========================================================
# 9) 실행
# =========================================================
print(f"[도메인{DOMAIN_NO}] 파일 로드: {DOMAIN_FILE}")
df_dom = load_domain_event(DOMAIN_FILE)

if "domain_label" not in df_dom.columns or df_dom["domain_label"].isna().all():
    df_dom["domain_label"] = DOMAIN_NO

# 0) ads_idx 정합성 점검
ads_mapping_issues = validate_ads_master_mapping(df_dom)

# 1) 상위 KPI
mda_kpi = build_mda_kpi(df_dom)

# 2) mda_reliable
mda_cut_table = build_auto_mda_cut_table(mda_kpi)
mda_kpi_rel = apply_mda_reliable(mda_kpi, mda_cut_table)

# 3) 상위 성과 세그먼트 + 운영 태그
mda_segmented = segment_mda_direct(mda_kpi_rel)

# 4) 보조 진단
mda_pubsub_detail = build_mda_pubsub_detail(df_dom)
mda_diag_summary = build_mda_diagnostic_summary(mda_pubsub_detail)

# 5) 메인 결과 + 진단 요약 결합
mda_segmented_full = mda_segmented.merge(
    mda_diag_summary,
    on=["domain_label", "ads_type", "mda_idx", "ads_type_name"],
    how="left"
)

# 6) 광고 단위 상세 KPI
mda_ads_detail = build_mda_ads_detail(df_dom)

# 7) 광고 단위 상세 + 매체사 분석 결과 결합
mda_ads_detail_with_segment = build_mda_ads_detail_with_segment(
    mda_ads_detail,
    mda_segmented_full
)

# 8) Top5
top5_perf = extract_top5_by_performance_segment(mda_segmented_full)
top5_op = extract_top5_by_operation_tag(mda_segmented_full)


# =========================================================
# 10) 저장
# =========================================================
mda_kpi_path = os.path.join(DOMAIN_OUT, f"domain{DOMAIN_NO}_mda_kpi.csv")
mda_cut_path = os.path.join(DOMAIN_OUT, f"domain{DOMAIN_NO}_mda_reliability_cut_selected.csv")
mda_segmented_path = os.path.join(DOMAIN_OUT, f"domain{DOMAIN_NO}_mda_segmented.csv")
mda_diag_summary_path = os.path.join(DOMAIN_OUT, f"domain{DOMAIN_NO}_mda_pubsub_diagnostic_summary.csv")
mda_pubsub_detail_path = os.path.join(DOMAIN_OUT, f"domain{DOMAIN_NO}_mda_pubsub_detail.csv")
top5_perf_path = os.path.join(DOMAIN_OUT, f"domain{DOMAIN_NO}_mda_top5_by_ads_type_and_performance_segment.csv")
top5_op_path = os.path.join(DOMAIN_OUT, f"domain{DOMAIN_NO}_mda_top5_by_ads_type_and_operation_tag.csv")

# 추가 파일
ads_issue_path = os.path.join(DOMAIN_OUT, f"domain{DOMAIN_NO}_ads_mapping_issues.csv")
mda_ads_detail_path = os.path.join(DOMAIN_OUT, f"domain{DOMAIN_NO}_mda_ads_detail.csv")
mda_ads_detail_seg_path = os.path.join(DOMAIN_OUT, f"domain{DOMAIN_NO}_mda_ads_detail_with_segment.csv")

mda_kpi_rel.to_csv(mda_kpi_path, index=False, encoding="utf-8-sig")
mda_cut_table.to_csv(mda_cut_path, index=False, encoding="utf-8-sig")
mda_segmented_full.to_csv(mda_segmented_path, index=False, encoding="utf-8-sig")
mda_diag_summary.to_csv(mda_diag_summary_path, index=False, encoding="utf-8-sig")
mda_pubsub_detail.to_csv(mda_pubsub_detail_path, index=False, encoding="utf-8-sig")
top5_perf.to_csv(top5_perf_path, index=False, encoding="utf-8-sig")
top5_op.to_csv(top5_op_path, index=False, encoding="utf-8-sig")

ads_mapping_issues.to_csv(ads_issue_path, index=False, encoding="utf-8-sig")
mda_ads_detail.to_csv(mda_ads_detail_path, index=False, encoding="utf-8-sig")
mda_ads_detail_with_segment.to_csv(mda_ads_detail_seg_path, index=False, encoding="utf-8-sig")

print("\n저장 완료:")
print(" -", mda_kpi_path)
print(" -", mda_cut_path)
print(" -", mda_segmented_path)
print(" -", mda_diag_summary_path)
print(" -", mda_pubsub_detail_path)
print(" -", top5_perf_path)
print(" -", top5_op_path)
print(" -", ads_issue_path)
print(" -", mda_ads_detail_path)
print(" -", mda_ads_detail_seg_path)

print("\n[성과 세그먼트 분포]")
print(mda_segmented_full["performance_segment"].value_counts(dropna=False))

print("\n[운영 태그 분포]")
print(mda_segmented_full["operation_tag"].value_counts(dropna=False))

print("\n[운영 태그 + 세부유형 분포]")
print(
    mda_segmented_full.groupby(["operation_tag", "operation_detail"], dropna=False)
    .size()
    .sort_values(ascending=False)
)

print("\n[광고유형별 × 운영 태그 Top5 샘플]")
print(top5_op.head(20))

print("\n[ads_idx 정합성 이슈 개수]")
print(len(ads_mapping_issues))

if len(ads_mapping_issues) > 0:
    print("\n[ads_idx 정합성 이슈 샘플]")
    print(ads_mapping_issues.head(20))
else:
    print("ads_idx 기준 ads_name / ads_type / mda_idx 정합성 이상 없음")

[도메인4] 파일 로드: D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\분석에 사용할 전처리 완료 최종 데이터 셋\도메인별 분류 테이블/domain_4.csv


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\내배캠 관련 모음\\최종 프로젝트 데이터 셋\\아이브 데이터 셋\\분석에 사용할 전처리 완료 최종 데이터 셋\\도메인별 분류 테이블/domain_4.csv'